# Tarea 13 · Filtrado avanzado de clientes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/13_filtrado_avanzado.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Matías.

Combinamos condiciones de contrato, servicio, cargos y antigüedad para identificar tres segmentos y comparar sus tasas de baja con la de toda la cartera.

## Cómo ejecutarlo

- **Local:** instalá las dependencias de `requirements.txt`, abrí el notebook desde el repo y ejecutá todas las celdas.
- **Colab:** ejecutá todas las celdas. El notebook lee los datos desde GitHub. El botón apunta a `main` y estará disponible cuando se integre la tarea. Para probarla antes, subí el archivo `.ipynb` a Colab.

## 1. Carga del dataset

Usamos `telco_clean.csv` si está disponible y `telco_provisorio.csv` mientras se completa la limpieza. Seleccionamos las columnas necesarias para construir los segmentos y medir las bajas.

In [ ]:
from pathlib import Path
from urllib.error import HTTPError

import pandas as pd

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

variables = ["Contract", "InternetService", "MonthlyCharges", "tenure", "PaymentMethod", "Churn"]
ruta_limpia = Path("data/processed/telco_clean.csv")
ruta_provisoria = Path("data/processed/telco_provisorio.csv")

if EN_COLAB:
    url_repo = "https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/"
    fuente = url_repo + ruta_limpia.as_posix()
    try:
        datos = pd.read_csv(fuente, usecols=variables)
    except HTTPError as error:
        if error.code != 404:
            raise
        fuente = url_repo + ruta_provisoria.as_posix()
        datos = pd.read_csv(fuente, usecols=variables)
else:
    actual = Path.cwd().resolve()
    raiz = next(
        (ruta for ruta in (actual, *actual.parents)
         if (ruta / ruta_limpia).is_file() or (ruta / ruta_provisoria).is_file()),
        None,
    )
    if raiz is None:
        raise FileNotFoundError("Falta el CSV de la tarea 3 o el dataset limpio de la tarea 9.")
    fuente = raiz / ruta_limpia
    if not fuente.is_file():
        fuente = raiz / ruta_provisoria
    datos = pd.read_csv(fuente, usecols=variables)

datos = datos[variables]
print(f"Fuente: {fuente}")

## 2. Consultas de negocio

`&` significa que deben cumplirse ambas condiciones. `|` significa que alcanza con que se cumpla una de ellas, aunque también pueden cumplirse ambas. Los paréntesis delimitan cada comparación y dejan explícito cómo se agrupan.

1. **Contrato mensual, fibra y cargo alto.** Seleccionamos clientes con las tres condiciones a la vez. Definimos cargo alto como un cargo mensual igual o superior al percentil 75 de la cartera, para estudiar clientes con cargos en el tramo superior.
2. **Clientes recientes con contrato mensual o cheque electrónico.** Buscamos clientes con hasta seis meses de antigüedad que además cumplan al menos una de las dos condiciones comerciales. Usamos seis meses como corte operativo para revisar las bajas durante el período inicial.
3. **Clientes con antigüedad y contrato de plazo fijo.** Buscamos clientes con al menos 24 meses de antigüedad y contrato de uno o dos años. Usamos este segmento para comparar la permanencia observada en clientes con una relación más larga y un compromiso de plazo fijo.

In [ ]:
assert datos["Churn"].isin(["Yes", "No"]).all(), "Churn debe contener Yes o No."
umbral_cargo = datos["MonthlyCharges"].quantile(0.75)

mensual_fibra_cargo_alto = (
    (datos["Contract"] == "Month-to-month")
    & (datos["InternetService"] == "Fiber optic")
    & (datos["MonthlyCharges"] >= umbral_cargo)
)
recientes_mensual_o_cheque = (
    (datos["tenure"] <= 6)
    & ((datos["Contract"] == "Month-to-month") | (datos["PaymentMethod"] == "Electronic check"))
)
antiguos_plazo_fijo = (
    (datos["tenure"] >= 24)
    & ((datos["Contract"] == "One year") | (datos["Contract"] == "Two year"))
)

print(f"Percentil 75 del cargo mensual: {umbral_cargo:.2f}")

## 3. Resultados

Para cada consulta contamos los clientes seleccionados y sus bajas. La tasa divide las bajas por los clientes del segmento. Agregamos la cartera completa como referencia. Las consultas se evalúan por separado y pueden compartir clientes.

In [ ]:
segmentos = {
    "Cartera completa": datos,
    "Mensual, fibra y cargo alto": datos.loc[mensual_fibra_cargo_alto],
    "Recientes, mensual o cheque electrónico": datos.loc[recientes_mensual_o_cheque],
    "Antiguos con plazo fijo": datos.loc[antiguos_plazo_fijo],
}
filas = []
for nombre, clientes in segmentos.items():
    bajas = clientes["Churn"].eq("Yes")
    filas.append({
        "Segmento": nombre,
        "Clientes": len(clientes),
        "Bajas": bajas.sum(),
        "Tasa de baja (%)": bajas.mean() * 100,
    })

resumen = pd.DataFrame(filas).set_index("Segmento")
resumen.round(2)

## 4. Interpretación con el CSV provisorio

- **Mensual, fibra y cargo alto.** Se dieron de baja 456 de 877 clientes (52,00 %), frente al 26,54 % de la cartera. El segmento combina una tasa alta con cargos mensuales de al menos 89,85.
- **Recientes, mensual o cheque electrónico.** Se dieron de baja 781 de 1.419 clientes (55,04 %). Es la mayor tasa de los tres filtros y muestra que las bajas son frecuentes en este grupo durante los primeros seis meses.
- **Antiguos con plazo fijo.** Se dieron de baja 185 de 2.725 clientes (6,79 %). Es un grupo grande con una tasa por debajo de la cartera. El filtro combina antigüedad y contrato, así que la diferencia observada no permite atribuir el resultado a una sola condición.

Revisar estas cifras cuando esté disponible `telco_clean.csv`.